#Simple TF-IDF

#all-MiniLM-L6-v2

In [ ]:
pip install rank_bm25

In [ ]:
"""
Advanced Product Search System
- Sentence Transformers (BERT-based semantic embeddings)
- Hybrid scoring (BM25 + Semantic + Intent)
- Query expansion with synonyms
- "Not found" detection
"""
import pandas as pd
import numpy as np
import re
from typing import List, Tuple, Dict
from dataclasses import dataclass
import pickle
import os

# Advanced NLP libraries
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import wordnet

# Download required NLTK data (run once)
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
    nltk.download('omw-1.4')

@dataclass
class SearchResult:
    product_id: str
    product_name: str
    category: str
    price: float
    description: str
    brand: str
    relevance_score: float
    matched_context: str
    explanation: str  # Why this was matched

class DataProcessor:
    """Auto-detects and processes any CSV dataset"""

    def __init__(self, csv_path: str):
        self.csv_path = csv_path
        self.df = None
        self.processed_df = None

    def load_data(self):
        print("📂 Loading dataset...")
        self.df = pd.read_csv(self.csv_path)
        print(f"✅ Loaded {len(self.df)} products\n")
        return self.df

    def clean_data(self):
        print("🧹 Processing data...")
        self.processed_df = self.df.copy()

        # Auto-detect columns
        column_mapping = {
            'product_name': ['name', 'product_name', 'title', 'product_title', 'product'],
            'category': ['category', 'product_category', 'product_category_tree'],
            'description': ['description', 'product_description', 'about_product'],
            'price': ['price', 'discounted_price', 'retail_price', 'selling_price'],
            'brand': ['brand', 'product_brand', 'manufacturer'],
        }

        renamed_cols = {}
        for standard_name, variations in column_mapping.items():
            for col in self.processed_df.columns:
                if col.lower() in [v.lower() for v in variations]:
                    if col != standard_name:
                        renamed_cols[col] = standard_name
                    break

        if renamed_cols:
            self.processed_df.rename(columns=renamed_cols, inplace=True)

        # Fill missing values
        for col in self.processed_df.columns:
            if self.processed_df[col].dtype == 'object':
                self.processed_df[col].fillna('', inplace=True)
            else:
                self.processed_df[col].fillna(0, inplace=True)

        # Create product ID
        if 'product_id' not in self.processed_df.columns:
            self.processed_df['product_id'] = self.processed_df.index.astype(str)

        # Create rich search text
        text_columns = []
        for col in ['product_name', 'description', 'category', 'brand']:
            if col in self.processed_df.columns:
                text_columns.append(col)

        self.processed_df['search_text'] = self.processed_df[text_columns].apply(
            lambda x: ' '.join(x.astype(str)), axis=1
        ).str.lower()

        # Normalize prices
        if 'price' in self.processed_df.columns:
            self.processed_df['price_numeric'] = self.processed_df['price'].apply(self._extract_price)

        print(f"✅ Data ready!\n")
        return self.processed_df

    def _extract_price(self, price_str):
        if pd.isna(price_str) or price_str == '':
            return 0.0
        price_str = str(price_str).replace('₹', '').replace(',', '').strip()
        match = re.search(r'\d+\.?\d*', price_str)
        return float(match.group()) if match else 0.0

class QueryExpander:
    """Expands queries with synonyms for better matching"""

    def expand_query(self, query: str) -> str:
        """Add synonyms to query"""
        words = query.lower().split()
        expanded_words = set(words)

        for word in words:
            # Get synonyms from WordNet
            synonyms = self._get_synonyms(word)
            expanded_words.update(synonyms[:2])  # Add top 2 synonyms

        return ' '.join(expanded_words)

    def _get_synonyms(self, word: str) -> List[str]:
        """Get synonyms using WordNet"""
        synonyms = []
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                synonym = lemma.name().replace('_', ' ')
                if synonym != word and synonym not in synonyms:
                    synonyms.append(synonym)
        return synonyms[:3]

class HybridSearchEngine:
    """
    Combines 3 search methods:
    1. Semantic Search (Sentence Transformers)
    2. BM25 (Keyword-based ranking)
    3. Intent-based scoring
    """

    def __init__(self):
        self.semantic_model = None
        self.embeddings = None
        self.bm25 = None
        self.tokenized_corpus = None

    def initialize(self, df: pd.DataFrame, use_cache: bool = True):
        """Initialize all search models"""
        print("⚙️  Initializing Hybrid Search Engine...\n")

        cache_file = 'embeddings_cache.pkl'

        # Try to load cached embeddings
        if use_cache and os.path.exists(cache_file):
            print("📦 Loading cached embeddings...")
            with open(cache_file, 'rb') as f:
                cache = pickle.load(f)
                self.embeddings = cache['embeddings']
                self.bm25 = cache['bm25']
                self.tokenized_corpus = cache['tokenized_corpus']
            print("✅ Loaded from cache\n")
        else:
            # Initialize Sentence Transformer (semantic embeddings)
            print("🤖 Loading Sentence Transformer model...")
            print("   Model: all-MiniLM-L6-v2 (fast & accurate)")
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

            # Create embeddings
            print("🔄 Creating semantic embeddings...")
            texts = df['search_text'].tolist()
            self.embeddings = self.semantic_model.encode(
                texts,
                show_progress_bar=True,
                batch_size=32
            )
            print(f"✅ Created {len(self.embeddings)} embeddings\n")

            # Initialize BM25
            print("📊 Initializing BM25 index...")
            self.tokenized_corpus = [doc.split() for doc in texts]
            self.bm25 = BM25Okapi(self.tokenized_corpus)
            print("✅ BM25 ready\n")

            # Save cache
            print("💾 Saving to cache...")
            with open(cache_file, 'wb') as f:
                pickle.dump({
                    'embeddings': self.embeddings,
                    'bm25': self.bm25,
                    'tokenized_corpus': self.tokenized_corpus
                }, f)
            print("✅ Cache saved\n")

    def search_semantic(self, query: str, top_k: int = 100) -> Tuple[np.ndarray, np.ndarray]:
        """Semantic search using sentence embeddings"""
        if self.semantic_model is None:
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

        query_embedding = self.semantic_model.encode([query])
        similarities = cosine_similarity(query_embedding, self.embeddings).flatten()

        top_indices = np.argsort(similarities)[::-1][:top_k]
        return top_indices, similarities[top_indices]

    def search_bm25(self, query: str, top_k: int = 100) -> Tuple[np.ndarray, np.ndarray]:
        """BM25 keyword-based search"""
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)

        top_indices = np.argsort(scores)[::-1][:top_k]
        return top_indices, scores[top_indices]

    def hybrid_search(self, query: str, top_k: int = 100) -> Tuple[np.ndarray, np.ndarray]:
        """
        Combine semantic and BM25 scores
        Weights: 70% Semantic + 30% BM25
        """
        # Get both rankings
        sem_indices, sem_scores = self.search_semantic(query, top_k=200)
        bm25_indices, bm25_scores = self.search_bm25(query, top_k=200)

        # Normalize scores to 0-1 range
        sem_scores_norm = sem_scores / (sem_scores.max() + 1e-10)
        bm25_scores_norm = bm25_scores / (bm25_scores.max() + 1e-10)

        # Combine scores
        combined_scores = {}

        for idx, score in zip(sem_indices, sem_scores_norm):
            combined_scores[idx] = score * 0.7  # 70% weight

        for idx, score in zip(bm25_indices, bm25_scores_norm):
            if idx in combined_scores:
                combined_scores[idx] += score * 0.3  # 30% weight
            else:
                combined_scores[idx] = score * 0.3

        # Sort and get top K
        sorted_items = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
        top_indices = np.array([idx for idx, _ in sorted_items[:top_k]])
        top_scores = np.array([score for _, score in sorted_items[:top_k]])

        return top_indices, top_scores

class IntentEngine:
    """Detect user intent and product categories"""

    def __init__(self):
        self.intent_contexts = {
            'TRAVEL_HOLIDAY': {
                'keywords': ['travel', 'trip', 'journey', 'vacation', 'holiday', 'tour', 'going'],
                'categories': ['luggage', 'bags', 'travel', 'suitcase', 'backpack', 'trolley'],
                'required_keywords': ['luggage', 'suitcase', 'bag', 'backpack', 'trolley', 'travel']
            },
            'FITNESS_GYM': {
                'keywords': ['gym', 'workout', 'fitness', 'exercise', 'training', 'running', 'sport'],
                'categories': ['fitness', 'sports', 'gym', 'shoes', 'clothing', 'equipment'],
                'required_keywords': ['gym', 'fitness', 'workout', 'sports', 'shoes', 'equipment']
            },
            'ELECTRONICS': {
                'keywords': ['laptop', 'phone', 'computer', 'tablet', 'electronics', 'gadget'],
                'categories': ['electronics', 'computer', 'mobile', 'accessories'],
                'required_keywords': ['laptop', 'computer', 'phone', 'mobile', 'tablet', 'electronic']
            },
            'CLOTHING_FASHION': {
                'keywords': ['clothes', 'fashion', 'wear', 'dress', 'shirt', 'pants', 'style'],
                'categories': ['clothing', 'fashion', 'apparel', 'wear'],
                'required_keywords': ['shirt', 'dress', 'pants', 'jean', 'clothing', 'apparel']
            },
            'HOME_KITCHEN': {
                'keywords': ['home', 'kitchen', 'furniture', 'decor', 'appliance'],
                'categories': ['home', 'kitchen', 'furniture', 'appliances'],
                'required_keywords': ['home', 'kitchen', 'furniture', 'appliance', 'decor']
            }
        }

    def detect_intent(self, query: str) -> List[Tuple[str, float]]:
        """Detect intent with confidence"""
        query_lower = query.lower()
        intent_scores = {}

        for intent_name, context in self.intent_contexts.items():
            score = 0.0
            matches = 0

            for keyword in context['keywords']:
                if re.search(r'\b' + keyword + r'\b', query_lower):
                    score += 2.0
                    matches += 1
                elif keyword in query_lower:
                    score += 1.0
                    matches += 1

            if matches > 0:
                confidence = min(score / (len(context['keywords']) * 2), 1.0)
                intent_scores[intent_name] = confidence

        sorted_intents = sorted(intent_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_intents if sorted_intents else [('GENERAL', 0.3)]

    def check_product_availability(self, query: str, df: pd.DataFrame) -> Tuple[bool, str]:
        """
        Check if products matching the query exist in dataset
        Returns: (has_products, message)
        """
        intents = self.detect_intent(query)

        if not intents or intents[0][1] < 0.3:
            return True, ""  # Unclear intent, let search proceed

        intent_name, confidence = intents[0]
        context = self.intent_contexts.get(intent_name, {})

        # Check if dataset has products in this category
        required_keywords = context.get('required_keywords', [])

        # Search in all text
        all_text = ' '.join(df['search_text'].tolist()).lower()

        found_keywords = [kw for kw in required_keywords if kw in all_text]

        if len(found_keywords) < 2:  # Need at least 2 relevant keywords
            return False, f"Sorry, we don't have {intent_name.replace('_', ' ').lower()} products in our catalog."

        return True, ""

class AdvancedSearchAgent:
    """Main search agent with hybrid approach"""

    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.hybrid_engine = HybridSearchEngine()
        self.intent_engine = IntentEngine()
        self.query_expander = QueryExpander()

    def initialize(self, use_cache: bool = True):
        """Initialize all components"""
        self.hybrid_engine.initialize(self.df, use_cache=use_cache)
        print("="*60)
        print("✅ ADVANCED SEARCH AGENT READY!")
        print("="*60)

    def search(self, query: str, top_n: int = 10) -> Tuple[List[SearchResult], str]:
        """
        Advanced search with multiple strategies
        Returns: (results, status_message)
        """
        # Check if products exist
        has_products, message = self.intent_engine.check_product_availability(query, self.df)

        if not has_products:
            return [], message

        # Detect intent
        intents = self.intent_engine.detect_intent(query)

        # Expand query with synonyms
        expanded_query = self.query_expander.expand_query(query)

        # Show what was detected
        if intents and intents[0][1] > 0.3:
            print(f"   🎯 Intent: {intents[0][0].replace('_', ' ').title()} ({intents[0][1]:.0%})")

        # Hybrid search
        indices, scores = self.hybrid_engine.hybrid_search(expanded_query, top_k=100)

        # Score with intent
        results = []
        for idx, base_score in zip(indices, scores):
            row = self.df.iloc[idx]

            final_score, context, explanation = self._calculate_relevance(
                row, intents, base_score, query
            )

            if final_score >= 0.25:  # Higher threshold
                result = SearchResult(
                    product_id=str(row.get('product_id', idx)),
                    product_name=str(row.get('product_name', 'Unknown'))[:100],
                    category=str(row.get('category', ''))[:50],
                    price=float(row.get('price_numeric', 0)),
                    description=str(row.get('description', ''))[:200],
                    brand=str(row.get('brand', 'Unknown'))[:50],
                    relevance_score=final_score,
                    matched_context=context,
                    explanation=explanation
                )
                results.append(result)

        results.sort(key=lambda x: x.relevance_score, reverse=True)

        if not results:
            return [], f"No products found matching '{query}'. Try different keywords."

        return results[:top_n], ""

    def _calculate_relevance(self, row, intents, base_score, query):
        """Calculate final relevance with explanation"""
        final_score = base_score * 0.5  # Start with hybrid score (50%)
        matched_context = 'GENERAL'
        explanation_parts = []

        # Intent boost (40%)
        if intents and intents[0][1] > 0.3:
            intent_name, confidence = intents[0]
            context = self.intent_engine.intent_contexts.get(intent_name, {})

            category_str = str(row.get('category', '')).lower()
            search_text = str(row.get('search_text', '')).lower()

            intent_boost = 0.0

            # Category match
            for cat in context.get('categories', []):
                if cat in category_str:
                    intent_boost += 0.6
                    matched_context = intent_name
                    explanation_parts.append(f"Category match: {cat}")
                    break

            # Keyword match
            found_keywords = [kw for kw in context.get('required_keywords', [])
                            if kw in search_text]
            if found_keywords:
                kw_score = len(found_keywords) / max(len(context.get('required_keywords', [])), 1)
                intent_boost += 0.4 * kw_score
                explanation_parts.append(f"Keywords: {', '.join(found_keywords[:3])}")

            final_score += intent_boost * confidence * 0.4

        # Direct query match (10%)
        query_terms = [t for t in query.lower().split() if len(t) > 2]
        search_text = str(row.get('search_text', '')).lower()
        matched_terms = [t for t in query_terms if t in search_text]

        if matched_terms:
            final_score += (len(matched_terms) / max(len(query_terms), 1)) * 0.1
            explanation_parts.append(f"Matched: {', '.join(matched_terms[:3])}")

        explanation = " | ".join(explanation_parts) if explanation_parts else "Semantic match"

        return min(final_score, 1.0), matched_context, explanation

def main():
    """Main execution"""
    print("="*70)
    print("  ADVANCED AI SEARCH SYSTEM")
    print("  • Sentence Transformers (BERT-based)")
    print("  • Hybrid BM25 + Semantic Search")
    print("  • Query Expansion with Synonyms")
    print("  • Smart 'Not Found' Detection")
    print("="*70)
    print()

    # ========== CHANGE THIS ==========
    csv_path = '/content/flipkart_com-ecommerce_sample 2.csv'
    # =================================

    if not os.path.exists(csv_path):
        print(f"❌ File not found: {csv_path}")
        return

    # Load data
    processor = DataProcessor(csv_path)
    df = processor.load_data()
    processed_df = processor.clean_data()

    # Initialize search
    agent = AdvancedSearchAgent(processor.processed_df)
    agent.initialize(use_cache=True)

    print("\n💬 Type your search queries (or 'quit' to exit)\n")

    while True:
        try:
            query = input("🔍 Search: ").strip()

            if query.lower() in ['quit', 'exit', 'q']:
                print("\n✅ Goodbye!")
                break

            if not query:
                continue

            print()
            results, message = agent.search(query, top_n=8)

            if message:
                print(f"ℹ️  {message}\n")
                continue

            if results:
                print(f"✨ Found {len(results)} products:\n")
                for i, result in enumerate(results, 1):
                    print(f"{i}. {result.product_name[:60]}")
                    print(f"   ₹{result.price:,.0f} | {result.brand} | {result.relevance_score:.0%} match")
                    print(f"   💡 {result.explanation}")
                    if result.description:
                        print(f"   📝 {result.description[:70]}...")
                print()

        except KeyboardInterrupt:
            print("\n\n✅ Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {e}\n")

if __name__ == "__main__":
    main()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


  ADVANCED AI SEARCH SYSTEM
  • Sentence Transformers (BERT-based)
  • Hybrid BM25 + Semantic Search
  • Query Expansion with Synonyms
  • Smart 'Not Found' Detection

📂 Loading dataset...
✅ Loaded 20000 products

🧹 Processing data...


/tmp/ipython-input-1850472452.py:83: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.processed_df[col].fillna('', inplace=True)
/tmp/ipython-input-1850472452.py:85: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

✅ Data ready!

⚙️  Initializing Hybrid Search Engine...

📦 Loading cached embeddings...
✅ Loaded from cache

✅ ADVANCED SEARCH AGENT READY!

💬 Type your search queries (or 'quit' to exit)

🔍 Search: i am going on holiday

✨ Found 8 products:

1. i-KitPit Pouch for Lenovo A369i
   ₹399 | i-KitPit | 35% match
   💡 Semantic match
   📝 Buy i-KitPit Pouch for Lenovo A369i only for Rs. 249 from Flipkart.com...
2. Mxofere Combo Orange Soap And Aloevera Lemon Facewash Kit
   ₹285 |  | 35% match
   💡 Semantic match
   📝 Buy Mxofere Combo Orange Soap And Aloevera Lemon Facewash Kit for Rs.2...
3. GIA Metal Necklace
   ₹999 | GIA | 35% match
   💡 Semantic match
   📝 GIA Metal Necklace - Buy GIA Metal Necklace only for Rs. 399 from Flip...
4. Mxofere Combo Orange Sandal Turmeric Jasmine Papaya Aloevera
   ₹295 |  | 34% match
   💡 Semantic match
   📝 Buy Mxofere Combo Orange Sandal Turmeric Jasmine Papaya Aloevera Soap ...
5. Gia Crystal Metal Necklace
   ₹4,499 | Gia | 33% match
   💡 Semantic matc